CTR Prediction Report

This assignment is about predicting if an ad gets clicked or not, using logistic regression trained by hand with SGD, on features like site, app, device, and hour.

First, the math part. The model predicts a probability p using a formula called sigmoid. Then we compare that probability to the real answer y (0 or 1, clicked or not) using something called log loss. When you do the calculus, the gradient (how much to change the weights) turns out to be really simple, its just p minus y.

In [ ]:
# p is the predicted probability, y is the real answer (0 or 1)
grad = p - y
# only update the weights that were actually used for this row
w[active_buckets] -= learning_rate * grad
b -= learning_rate * grad

The reason this matters is that each row only turns on a few buckets out of possibly millions (because of hashing, explained below). So most of the features x are 0. That means the gradient is also mostly 0, so each update only touches a few weights, not all of them. Thats why this can scale to huge numbers of features.

Next, the hashing trick. Real ad data has way too many possible values (like millions of device IDs) to just make one column per value. So instead we hash each column-value pair into one of D buckets.

In [ ]:
import hashlib
def hash_row(row, cols, D):
    idxs = []
    for c in cols:
        token = f"{c}={row[c]}"
        h = int(hashlib.md5(token.encode()).hexdigest(), 16)
        idxs.append(h % D)
    return idxs

You don’t have to worry about running out of memory and its size will be constant regardless of the number of unique values you may want to store. However, another problem is that you run the risk of multiple values ending up in the same bucket; this is known as a collision and these can become jumbled together.

Results: if we simply assume an average click rate on every instance of our model we can expect a log loss of 0.4503. My model was able to achieve a log loss of 0.4124 and my average precision was .342 (which is approximately .17 for random guesses).

Accuracy isn’t a good metric to evaluate your performance because most ads do not receive any clicks — roughly 17% — so a model that predicts “no click” all the time achieves an accuracy of 83%, yet has no value. The log loss and average precision are far superior measures of performance because they measure whether or not the model is capable of predicting which instances are likely to result in a click versus those that won’t.

I also wanted to experiment with what kind of impact using larger or smaller “buckets” (the parameter D) had on my model’s ability to predict clicks:

D = 256, Average Precision = 0.264, 
D = 4096, Average Precision = 0.324, 
D = 16384, Average Precision = 0.338, 
D = 65536, Average Precision = 0.342, 
D = 262144, Average Precision = 0.342, 

The average precision continued to increase from D = 256 through D = 16384. At that point it began to plateau and improve very little. Prior to that point there were too many values competing for placement within the same bucket and therefore could easily collide with one another. After that point the occurrence of collisions became sufficiently rare such that increasing D did not provide additional benefit and instead increased memory usage without providing any advantage.

The last portion of evaluation concerns calibration; essentially, calibration refers to testing whether the probability that you assign to something occurring (i.e., an actual click) matches the true likelihood of it happening. In the case of ad auctions, since the bid amount is directly proportional to the predicted probability assigned to the likelihood of a click occurring, incorrect predictions can lead to incorrect bids even if the ordering of advertisements is correct. Overall, my calibration curve appeared fairly accurate for both low- and mid-range probabilities, however, I experienced issues at higher ranges due to less data available for validation.